In [16]:
import pandas as pd

In [17]:
a ="https://learn.microsoft.com/en-us/windows-server/identity/ad-ds/plan/appendix-l--events-to-monitor"

In [18]:
tables = pd.read_html(a)

In [19]:
df = tables[0]

In [20]:
df


,Current Windows Event ID,Legacy Windows Event ID,Potential Criticality,Event Summary
0,4618,NaN,High,A monitored security event pattern has occurred.
1,4649,NaN,High,A replay attack was detected. May be a harmles...
2,4719,612,High,System audit policy was changed.
3,4765,NaN,High,SID History was added to an account.
4,4766,NaN,High,An attempt to add SID History to an account fa...
...,...,...,...,...
376,24588,NaN,Low,The conversion operation on volume %2 encounte...
377,24595,NaN,Low,Volume %2 contains bad clusters. These cluster...
378,24621,NaN,Low,Initial state check: Rolling volume conversion...
379,5049,NaN,Low,An IPsec Security Association was deleted.


In [21]:
high = df[df["Potential Criticality"] == "High"].copy()
low = df[df["Potential Criticality"] == "Low"].copy()
medium = df[df["Potential Criticality"] == "Medium"].copy()
Medium_to_High = df[df["Potential Criticality"] == "Medium to High"].copy()

low['Current Windows Event ID'] = pd.to_numeric(low['Current Windows Event ID'], errors='coerce')
medium['Current Windows Event ID'] = pd.to_numeric(medium['Current Windows Event ID'], errors='coerce')
Medium_to_High['Current Windows Event ID'] = pd.to_numeric(Medium_to_High['Current Windows Event ID'], errors='coerce')
high['Current Windows Event ID'] = pd.to_numeric(high['Current Windows Event ID'], errors='coerce')

In [22]:


def scoring2(df1):
    import pandas as pd

    # ---------- STEP 1: Convert EventID to numeric ----------
    df1['EventID'] = pd.to_numeric(df1['EventID'], errors='coerce')

    # ---------- STEP 2: Base Microsoft severity scoring ----------
    df1['suspicious_score'] = 0.0
    df1.loc[df1['EventID'].isin(low['Current Windows Event ID']), 'suspicious_score'] = 1.0
    df1.loc[df1['EventID'].isin(medium['Current Windows Event ID']), 'suspicious_score'] = 2.0
    df1.loc[df1['EventID'].isin(Medium_to_High['Current Windows Event ID']), 'suspicious_score'] = 2.5
    df1.loc[df1['EventID'].isin(high['Current Windows Event ID']), 'suspicious_score'] = 3.0

    

   #  Remote Desktop logons 
    df1.loc[df1['LogonType'] == 10, 'suspicious_score'] += 1.0

    # Suspicious process execution
    risky_processes = [
        "powershell.exe", "cmd.exe", "wmic.exe", "mimikatz.exe",
        "rundll32.exe", "bitsadmin.exe"
    ]
    df1.loc[df1['NewProcessName'].str.lower().isin([p.lower() for p in risky_processes]), 'suspicious_score'] += 1.0

    # Rule: Account privilege changes
    privileged_events = [4720, 4728, 4729, 4732, 4733]
    df1.loc[df1['EventID'].isin(privileged_events), 'suspicious_score'] += 2.0

    # Rule: Service or task creation
    service_events = [4697, 4702]
    df1.loc[df1['EventID'].isin(service_events), 'suspicious_score'] += 1.0

    # Rule: External source IPs
    def is_external(ip):
        if pd.isna(ip): return False
        return not (
            ip.startswith("10.") or
            ip.startswith("192.168.") or
            any(ip.startswith(f"172.{i}.") for i in range(16, 32))
        )
    df1.loc[df1['SourceAddress'].apply(is_external), 'suspicious_score'] += 0.5

    # Optional: Cap the score to a maximum
    df1['suspicious_score'] = df1['suspicious_score'].clip(upper=5.0)

    return df1

In [ ]:
# ---------------------------
# Extra scoring rules
# ---------------------------

# Failed logon attempts
failed_logon = [4625]
df1.loc[df1["EventID"].isin(failed_logon), "suspicious_score"] += 0.5


# Log cleared (very suspicious)
log_cleared = [1102]
df1.loc[df1["EventID"].isin(log_cleared), "suspicious_score"] += 3.0


# Account created
account_created = [4720]
df1.loc[df1["EventID"].isin(account_created), "suspicious_score"] += 2.0


# Added to admin / privileged group
admin_events = [4728, 4732, 4756]
df1.loc[df1["EventID"].isin(admin_events), "suspicious_score"] += 2.0


# Scheduled task created
task_events = [4698]
df1.loc[df1["EventID"].isin(task_events), "suspicious_score"] += 1.5


# Service installed
service_events = [4697]
df1.loc[df1["EventID"].isin(service_events), "suspicious_score"] += 1.0


# Remote Desktop logon
df1.loc[df1["LogonType"] == 10, "suspicious_score"] += 1.0


# Network logon
df1.loc[df1["LogonType"] == 3, "suspicious_score"] += 0.3


# Suspicious processes
risky_processes = [
    "powershell.exe",
    "cmd.exe",
    "wmic.exe",
    "mimikatz.exe",
    "rundll32.exe",
    "bitsadmin.exe"
]

df1.loc[
    df1["NewProcessName"].str.lower().isin(risky_processes),
    "suspicious_score"
] += 1.0


# Process from temp folder
df1.loc[
    df1["NewProcessName"].str.contains("temp", case=False, na=False),
    "suspicious_score"
] += 1.0


# SYSTEM account activity
df1.loc[
    df1["SubjectUserName"] == "SYSTEM",
    "suspicious_score"
] += 0.5


# External IP
def is_external(ip):
    if pd.isna(ip):
        return False
    return not (
        ip.startswith("10.") or
        ip.startswith("192.168.") or
        any(ip.startswith(f"172.{i}.") for i in range(16, 32))
    )

df1.loc[
    df1["SourceAddress"].apply(is_external),
    "suspicious_score"
] += 0.5


# Cap score
df1["suspicious_score"] = df1["suspicious_score"].clip(upper=5.0)

In [ ]:
# maybe use this one def scoring_defensible(df1):
    import pandas as pd

    # ------------------------
    # Step 1: Base Microsoft severity scoring
    # ------------------------
    df1['EventID'] = pd.to_numeric(df1['EventID'], errors='coerce')
    df1['suspicious_score'] = 0.0

    # Keep the Microsoft severity scoring exactly as your original
    df1.loc[df1['EventID'].isin(low['Current Windows Event ID']), 'suspicious_score'] = 1.0
    df1.loc[df1['EventID'].isin(medium['Current Windows Event ID']), 'suspicious_score'] = 2.0
    df1.loc[df1['EventID'].isin(Medium_to_High['Current Windows Event ID']), 'suspicious_score'] = 2.5
    df1.loc[df1['EventID'].isin(high['Current Windows Event ID']), 'suspicious_score'] = 3.0

    # ------------------------
    # Step 2: Exceptional extra scoring
    # ------------------------

    # Remote Desktop logon (RDP)
    df1.loc[df1['LogonType'] == 10, 'suspicious_score'] += 1.0

    # Suspicious processes (not already reflected in base)
    risky_processes = ["powershell.exe", "cmd.exe", "wmic.exe", "mimikatz.exe",
                       "rundll32.exe", "bitsadmin.exe"]
    df1.loc[df1['NewProcessName'].str.lower().isin([p.lower() for p in risky_processes]), 'suspicious_score'] += 1.0

    # Process executed from temp folder
    df1.loc[df1['NewProcessName'].str.contains("temp", case=False, na=False), 'suspicious_score'] += 1.0

    # SYSTEM account activity
    df1.loc[df1['SubjectUserName'] == "SYSTEM", 'suspicious_score'] += 0.5

    # Log cleared (highly suspicious)
    df1.loc[df1['EventID'] == 1102, 'suspicious_score'] += 3.0

    # External IP access
    def is_external(ip):
        if pd.isna(ip):
            return False
        return not (
            ip.startswith("10.") or
            ip.startswith("192.168.") or
            any(ip.startswith(f"172.{i}.") for i in range(16, 32))
        )
    df1.loc[df1['SourceAddress'].apply(is_external), 'suspicious_score'] += 0.5

    # ------------------------
    # Step 3: Cap the score
    # ------------------------
    df1['suspicious_score'] = df1['suspicious_score'].clip(upper=5.0)

    return df1

In [23]:
import Evtx.Evtx as evtx

In [24]:
import csv
from Evtx.Evtx import Evtx
import xml.etree.ElementTree as ET

# Input and output
input_file = "DE_RDP_Tunnel_5156.evtx"
output_file = "output_flat.csv"

# XML namespace
ns = {"e": "http://schemas.microsoft.com/win/2004/08/events/event"}

with Evtx(input_file) as log:
    # Collects field_names
    field_names = set(["RecordNumber", "Timestamp", "EventID"])

    for record in log.records():
        xml_data = record.xml()
        root = ET.fromstring(xml_data)
        for data in root.findall(".//e:Data", ns):
            field_names.add(data.attrib.get("Name", "Unknown"))

    field_names = list(field_names)  # convert to list for writing CSV

    #writes CSV file
    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=field_names)
        writer.writeheader()
# get all the data and writes it 
        with Evtx(input_file) as log2:
            for record in log2.records():
                xml_data = record.xml()
                root = ET.fromstring(xml_data)
                
                event = {
                    "RecordNumber": record.record_num(),
                    "Timestamp": root.find(".//e:TimeCreated", ns).attrib.get("SystemTime", ""),
                    "EventID": root.find(".//e:EventID", ns).text if root.find(".//e:EventID", ns) is not None else ""
                }

                # Add all EventData fields
                for data in root.findall(".//e:Data", ns):
                    field = data.attrib.get("Name", "Unknown")
                    value = data.text if data.text else ""
                    event[field] = value

                writer.writerow(event)

print("Flat CSV created:", output_file)

Flat CSV created: output_flat.csv


In [4]:
# import pandas as pd

# df = pd.read_csv(r"C:\Users\tcsim\Downloads\On_demand_report_2026-04-01T21_53_10.828Z_2cef7ec0-2e15-11f1-9076-7970bfa6d5ac.csv")
df.head(10)
# scored = scoring2(df)

,agent\.name,agent\.ip,rule\.id,rule\.description,data\.vulnerability\.severity,location,data\.win\.system\.severityValue,data\.win\.system\.systemTime,rule\.info,full_log
0,NormalLogs,192.168.80.140,23505,CVE-2019-15316 affects Steam,High,vulnerability-detector,,,,
1,NormalLogs,192.168.80.140,23505,CVE-2019-17180 affects Steam,High,vulnerability-detector,,,,
2,NormalLogs,192.168.80.140,23505,CVE-2020-15530 affects Steam,High,vulnerability-detector,,,,
3,NormalLogs,192.168.80.140,23505,CVE-2019-15315 affects Steam,High,vulnerability-detector,,,,
4,NormalLogs,192.168.80.140,23504,CVE-2019-14743 affects Steam,Medium,vulnerability-detector,,,,
5,NormalLogs,192.168.80.140,60642,Software protection service scheduled successf...,,EventChannel,INFORMATION,2026-04-01T21:24:02.3267917Z,,
6,NormalLogs,192.168.80.140,60642,Software protection service scheduled successf...,,EventChannel,INFORMATION,2026-04-01T21:19:26.8328324Z,,
7,wazuh-server,,19012,CIS Benchmark for Amazon Linux 2023 Benchmark ...,,sca,,,,
8,wazuh-server,,19009,CIS Benchmark for Amazon Linux 2023 Benchmark ...,,sca,,,,
9,NormalLogs,192.168.80.140,61104,Service startup type was changed,,EventChannel,INFORMATION,2026-04-01T19:55:18.6380822Z,This does not appear to be logged on Windows 2000,


In [26]:
scored

,SubjectDomainName,SubjectUserSid,WorkstationName,NewProcessName,LayerName,DestAddress,SourcePort,TargetUserName,ProcessName,PrivilegeList,...,Application,TargetDomainName,TargetInfo,ProcessId,LmPackageName,TokenElevationType,SourceAddress,IpAddress,TargetLogonGuid,suspicious_score
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.5
1,NaN,NaN,NaN,NaN,%%14611,ff02::1:2,546.0,NaN,NaN,NaN,...,\device\harddiskvolume1\windows\system32\svcho...,NaN,NaN,NaN,NaN,NaN,fe80::80ac:4126:fa58:1b81,NaN,NaN,1.5
2,EXAMPLE,S-1-5-18,NaN,C:\Windows\System32\TSTheme.exe,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0x00000278,NaN,%%1936,NaN,NaN,NaN,1.0
3,NaN,NaN,NaN,NaN,%%14611,10.0.2.15,49263.0,NaN,NaN,NaN,...,\device\harddiskvolume1\windows\system32\lsass...,NaN,NaN,NaN,NaN,NaN,10.0.2.17,NaN,NaN,1.0
4,EXAMPLE,S-1-5-18,NaN,NaN,NaN,NaN,NaN,user01,C:\Windows\System32\winlogon.exe,NaN,...,NaN,EXAMPLE,localhost,0x00000704,NaN,NaN,NaN,127.0.0.1,{00000000-0000-0000-0000-000000000000},1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,NaN,NaN,NaN,NaN,%%14610,127.0.0.1,1900.0,NaN,NaN,NaN,...,\device\harddiskvolume1\windows\system32\svcho...,NaN,NaN,NaN,NaN,NaN,239.255.255.250,NaN,NaN,1.5
97,NaN,NaN,NaN,NaN,%%14610,::1,1900.0,NaN,NaN,NaN,...,\device\harddiskvolume1\windows\system32\svcho...,NaN,NaN,NaN,NaN,NaN,ff02::c,NaN,NaN,1.5
98,NaN,NaN,NaN,NaN,%%14610,127.0.0.1,1900.0,NaN,NaN,NaN,...,\device\harddiskvolume1\windows\system32\svcho...,NaN,NaN,NaN,NaN,NaN,239.255.255.250,NaN,NaN,1.5
99,NaN,NaN,NaN,NaN,%%14610,::1,1900.0,NaN,NaN,NaN,...,\device\harddiskvolume1\windows\system32\svcho...,NaN,NaN,NaN,NaN,NaN,ff02::c,NaN,NaN,1.5


In [27]:
# scored.to_csv("Dataset1_scored.csv", index=False)

In [28]:
df.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
96     False
97     False
98     False
99     False
100    False
Length: 101, dtype: bool